In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [9]:
import numpy as np
import pandas as pd
import torch
import os
import gc
import shutil
import logging
from dataclasses import dataclass
from typing import Optional, Union
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    logging as hf_logging,
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

In [10]:
# ================= SILENCE NOISY (HARMLESS) WARNINGS =================
hf_logging.set_verbosity_error()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [11]:
# ================= DEVICE =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print("DEVICE:", device, "| bf16 supported:", use_bf16)

DEVICE: cuda | bf16 supported: True


In [12]:
# ================= SETUP =================
model_name = "roberta-large"
FOLDS = 4
MAX_LEN = 320     
SEED = 42

In [13]:
# ================= LOAD DATA =================
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

label_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
inv_map = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}
train_df["label"] = train_df["answer"].map(label_map)

OPTION_COLS = ["A", "B", "C", "D", "E"]

In [14]:
# ================= TOKENIZER & PREPROCESSING =================
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples, option_order=OPTION_COLS):
    first_sentences = [[context] * 5 for context in examples["prompt"]]
    second_sentences = []
    for i in range(len(examples["prompt"])):
        options = [str(examples[col][i]) for col in option_order]
        second_sentences.append(options)

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, second_sentences, truncation=True, max_length=MAX_LEN,
    )

    features = {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}
    return features

print("Preparing test data (standard option order)...")
test_ds = Dataset.from_pandas(test_df)
test_ds_mapped = test_ds.map(preprocess_function, batched=True,
                              remove_columns=[c for c in test_ds.column_names if c != 'id'])

REVERSED_COLS = OPTION_COLS[::-1]
print("Preparing test data (reversed option order, for TTA)...")
test_ds_rev_mapped = test_ds.map(
    lambda ex: preprocess_function(ex, option_order=REVERSED_COLS),
    batched=True, remove_columns=[c for c in test_ds.column_names if c != 'id']
)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Preparing test data (standard option order)...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Preparing test data (reversed option order, for TTA)...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [15]:
# ================= CUSTOM DATA COLLATOR =================
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features] if label_name in features[0].keys() else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])

        flattened_features = [[{k: v[i] for k, v in feature.items() if k != 'id'} for i in range(num_choices)] for feature in features]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features, padding=self.padding, max_length=self.max_length, pad_to_multiple_of=self.pad_to_multiple_of, return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

In [16]:
# ================= METRIC: MAP@3 (matches the top-3 submission format) =================
def map_at_3(labels, logits):
    top3 = np.argsort(-logits, axis=1)[:, :3]
    score = 0.0
    for i, label in enumerate(labels):
        if label == top3[i, 0]:
            score += 1.0
        elif label == top3[i, 1]:
            score += 1.0 / 2
        elif label == top3[i, 2]:
            score += 1.0 / 3
    return score / len(labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "map3": map_at_3(labels, logits),
    }

In [17]:
# ================= K-FOLD TRAINING LOOP =================
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
 
all_test_preds = []       
all_test_preds_rev = []   
fold_weights = []       
 
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label"])):
    fold_dir = f"./roberta_fold_{fold}"
    print(f"\n{'='*20} FOLD {fold + 1}/{FOLDS} {'='*20}")
 
    train_data = train_df.iloc[train_idx].reset_index(drop=True)
    val_data = train_df.iloc[val_idx].reset_index(drop=True)
 
    train_ds = Dataset.from_pandas(train_data)
    val_ds = Dataset.from_pandas(val_data)
 
    train_ds = train_ds.map(preprocess_function, batched=True, remove_columns=train_ds.column_names)
    val_ds = val_ds.map(preprocess_function, batched=True, remove_columns=val_ds.column_names)
 
    train_ds = train_ds.add_column("label", train_data["label"].tolist())
    val_ds = val_ds.add_column("label", val_data["label"].tolist())
 
    model = AutoModelForMultipleChoice.from_pretrained(model_name).to(device)
 
    training_args = TrainingArguments(
        output_dir=fold_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
 
        save_only_model=True,
        save_total_limit=1,
 
        learning_rate=1e-5,             
        weight_decay=0.01,
        num_train_epochs=4,             
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        warmup_ratio=0.15,              
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        label_smoothing_factor=0.05,    
        logging_steps=20,
        bf16=use_bf16,                  
        fp16=not use_bf16,
        load_best_model_at_end=True,
        metric_for_best_model="map3",   
        greater_is_better=True,
        report_to="none",
        seed=SEED,
    )
 
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
 
    trainer.train()
 
    val_metrics = trainer.evaluate()
    fold_map3 = val_metrics["eval_map3"]
    fold_weights.append(fold_map3)
    print(f"Fold {fold + 1} val MAP@3: {fold_map3:.4f} | val accuracy: {val_metrics['eval_accuracy']:.4f}")
 
    print(f"Inferencing test data for fold {fold + 1} (standard order)...")
    fold_preds = trainer.predict(test_ds_mapped).predictions
    all_test_preds.append(fold_preds)
 
    print(f"Inferencing test data for fold {fold + 1} (reversed order, TTA)...")
    fold_preds_rev = trainer.predict(test_ds_rev_mapped).predictions
    # un-reverse columns so index 0 is always option A again
    fold_preds_rev = fold_preds_rev[:, ::-1]
    all_test_preds_rev.append(fold_preds_rev)
 
    print(f"Cleaning up disk and memory for fold {fold + 1}...")
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()
    if os.path.exists(fold_dir):
        shutil.rmtree(fold_dir)


==================== FOLD 1/4 ====================


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

{'loss': '25.09', 'grad_norm': '522.6', 'learning_rate': '5.933e-06', 'epoch': '2.128'}
{'loss': '24.92', 'grad_norm': '1004', 'learning_rate': '3.97e-06', 'epoch': '2.555'}
{'loss': '24.8', 'grad_norm': '1360', 'learning_rate': '2.166e-06', 'epoch': '2.981'}
{'eval_loss': '2.982', 'eval_accuracy': '0.454', 'eval_map3': '0.6', 'eval_runtime': '34.54', 'eval_samples_per_second': '14.48', 'eval_steps_per_second': '1.824', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '24.47', 'grad_norm': '1064', 'learning_rate': '7.986e-07', 'epoch': '3.405'}
{'loss': '24.69', 'grad_norm': '505.4', 'learning_rate': '7.885e-08', 'epoch': '3.832'}
{'eval_loss': '2.96', 'eval_accuracy': '0.456', 'eval_map3': '0.596', 'eval_runtime': '34.75', 'eval_samples_per_second': '14.39', 'eval_steps_per_second': '1.813', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1729', 'train_samples_per_second': '3.47', 'train_steps_per_second': '0.109', 'train_loss': '25.12', 'epoch': '4'}
{'eval_loss': '2.983', 'eval_accuracy': '0.45', 'eval_map3': '0.5953', 'eval_runtime': '34.95', 'eval_samples_per_second': '14.31', 'eval_steps_per_second': '1.803', 'epoch': '4'}
Fold 4 val MAP@3: 0.5953 | val accuracy: 0.4500
Inferencing test data for fold 4 (standard order)...
Inferencing test data for fold 4 (reversed order, TTA)...
Cleaning up disk and memory for fold 4...


In [20]:
# ================= ENSEMBLE & FINAL OUTPUT =================
print("\n" + "="*40)
print("Fold MAP@3 scores:", [f"{w:.4f}" for w in fold_weights])

fold_weights = np.array(fold_weights)
fold_weights = fold_weights / fold_weights.sum()  

# softmax each fold's logits before combining (both orderings), then weighted-average probabilities
std_probs = [torch.softmax(torch.tensor(p), dim=1).numpy() for p in all_test_preds]
rev_probs = [torch.softmax(torch.tensor(p.copy()), dim=1).numpy() for p in all_test_preds_rev]

# average the two TTA views per fold first, then weight across folds
per_fold_probs = [(s + r) / 2 for s, r in zip(std_probs, rev_probs)]
final_probs = np.tensordot(fold_weights, np.stack(per_fold_probs), axes=(0, 0))

final = []
for i, p in enumerate(final_probs):
    top3_indices = np.argsort(p)[::-1][:3]
    top3_letters = [inv_map[idx] for idx in top3_indices]
    final.append({
        "id": test_ds["id"][i],
        "prediction": " ".join(top3_letters)
    })

submission = pd.DataFrame(final)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print(submission.head())
print("Ensemble submission ready.")


Fold MAP@3 scores: ['0.3078', '0.3125', '0.1871', '0.1926']
   id prediction
0   1      A B D
1   2      B D A
2   3      B E C
3   4      E C A
4   5      C D A
Ensemble submission ready.
